# Stage 6c — QUICK comparison on a test subsample (no full run)

Answers three questions in one session, using a **subsample of the test split** instead of the
2h+ full pass:

1. **Which HF repo is which? — RESOLVED (25 Jul, via HF API):** the account has exactly 3 model repos:
   `llada-moe-lora-run2-ckpt` (Jul 16, `latest/` only), `llada-moe-lora-run2-NYC-hyperbolic-ckpt`
   (Jul 18-19, `best/`+`latest/`), and `llada-moe-lora-run2-NYC-hyperbolic-h30-ckpt` (Jul 19, **empty**
   — the HIST_LEN=30 retrain that died before its first push). Same LoRA architecture in both trained
   repos, but the Jul-18 retrain added overfit guards (weight decay 0.01/0.1 + head dropout 0.1) — and
   scores WORSE (full-val Acc@1 0.1594 vs 0.1809). `run2-ckpt` also has the only complete FULL-test
   eval: Acc@1=0.1699 / MRR=0.2508 (n=29,071, Jul 17) — the anchor to compare this notebook's subsample
   rows against. **No bare (run-1) checkpoint exists on HF**, so the comparison below runs as eval-time
   ablations on the trained checkpoint (§0b re-verifies the inventory any time).
2. **Comparison table** for the three upgrades, applied cumulatively at eval time:
   80/10/10 stage-0 split → 70/10/20 re-split → + user profile in the prompt → + trained output layer.
3. Optionally the same rows for a second checkpoint (edit `CKPTS`).

**Cost:** ~9 min per table row at `TEST_MAX=2000`, `BATCH_SIZE=8` on a T4 (≈4.5 min at batch 16).
Default grid = 4 rows ≈ 35 min.

**Setup (same as Stage 6b):** attach the stage-0 Kaggle dataset (`train/val/test_NYC.csv`,
`poi_metadata_NYC.csv`, `vocab.pkl`, `poi_hyperbolic_embs.npy`), add `HF_TOKEN` via Add-ons → Secrets,
GPU on. Run §0a FIRST in a fresh kernel.

⚠ **How to read the table (important):** these are *eval-time* toggles on a checkpoint that was
*trained* with the full config. Rows without the profile / with the stage-0 split measure how the
trained model behaves under those inputs — a fair quick estimate of each ingredient's contribution,
but not identical to retraining without it. For the `lm_head` (no-output-layer) rows note that the full
checkpoint's lm_head POI rows were never trained, so those rows are expected to be ~random
(≈1/5120 = 0.0002) — which is itself the cleanest evidence that the output layer does the ranking
work. Since no bare checkpoint was ever uploaded, this eval-time version IS the available comparison.


In [ ]:
# ── 0a · Auth + pre-download BEFORE the transformers pin (fresh kernel!) ────
import os, sys, time

os.environ["HF_HUB_DISABLE_XET"] = "1"
if "huggingface_hub" in sys.modules:
    print("⚠ huggingface_hub already imported — restart the session and run this cell FIRST.")

if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"⚠ could not load HF_TOKEN from Kaggle Secrets ({type(e).__name__}: {e})")

from huggingface_hub import snapshot_download, whoami
print("HF auth:", whoami()["name"] if os.environ.get("HF_TOKEN") else "ANONYMOUS (rate-limited!)")

MODEL_NAME_0A = "inclusionAI/LLaDA-MoE-7B-A1B-Instruct"
_t0 = time.time()
MODEL_PATH = snapshot_download(MODEL_NAME_0A, max_workers=4)
print(f"base snapshot ready in {time.time()-_t0:.0f}s -> {MODEL_PATH}")

# ── checkpoints to compare ──────────────────────────────────────────────────
# key -> (repo, subfolder). Verified 25 Jul: only two trained repos exist, same LoRA config;
# no bare checkpoint was ever uploaded. Uncomment the second line to also compare the earlier job.
CKPTS_0A = {
    # run2-ckpt latest/ = the better checkpoint (full-test Acc@1=0.1699) — use it for the grid.
    "full": ("yosrr12/llada-moe-lora-run2-ckpt", "latest"),
    # "guards": ("yosrr12/llada-moe-lora-run2-NYC-hyperbolic-ckpt", "best"),  # Jul-18 retrain w/ overfit guards (val Acc@1 0.1594)
}

CKPT_DIRS = {}
for key, (repo, sub) in CKPTS_0A.items():
    root = snapshot_download(repo, allow_patterns=[f"{sub}/*"],
                             ignore_patterns=["*trainer_state*"], max_workers=4)
    d = os.path.join(root, sub)
    has_head = os.path.isfile(os.path.join(d, "poi_head.pt"))
    assert os.path.isfile(os.path.join(d, "adapter_model.safetensors")), f"{repo}/{sub}: no adapter!"
    CKPT_DIRS[key] = d
    print(f"[{key}] {repo}/{sub}  poi_head.pt={'YES (has output layer)' if has_head else 'NO (bare)'}")


In [ ]:
# ── 0b · WHICH REPO IS WHICH? — list every model repo under your account ────
# The bare (run-1) experiment = 80/10/10, no profile, NO output layer -> no poi_head.pt.
# The full experiment = 70/10/20 re-split + profile + output layer -> has poi_head.pt.
from huggingface_hub import HfApi
api = HfApi()
me = api.whoami()["name"]
print(f"account: {me}\n")
rows = []
for m in api.list_models(author=me):
    try:
        files = api.list_repo_files(m.id)
    except Exception as e:
        print(f"{m.id}: <cannot list: {e}>"); continue
    subs  = sorted({f.split("/")[0] for f in files if "/" in f})
    heads = sorted({f for f in files if f.endswith("poi_head.pt")})
    info  = api.model_info(m.id)
    rows.append((m.id, ",".join(subs) or "-", "YES" if heads else "no",
                 str(info.last_modified)[:16] if info.last_modified else "?"))
print(f"{'repo':58s} {'subfolders':22s} {'poi_head?':9s} last_modified")
for r in sorted(rows, key=lambda r: r[3]):
    print(f"{r[0]:58s} {r[1]:22s} {r[2]:9s} {r[3]}")
print("\nExpected (verified 25 Jul): run2-ckpt (Jul 16, latest/), run2-NYC-hyperbolic-ckpt")
print("(Jul 18-19, best/+latest/) — both WITH poi_head.pt, identical LoRA configs; and the empty")
print("aborted run2-NYC-hyperbolic-h30-ckpt. No bare checkpoint exists on HF.")


In [ ]:
# ── 0c · Hard-clean install of the pinned stack (after §0a!) ────────────────
import glob, shutil, os
!pip uninstall -y -q transformers tokenizers 2>/dev/null
for pat in ("transformers", "transformers-*", "tokenizers", "tokenizers-*"):
    for p in glob.glob(f"/usr/local/lib/python3.12/dist-packages/{pat}"):
        shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
!pip install -q --no-cache-dir "transformers==4.46.3" "bitsandbytes>=0.46.1" "peft==0.13.2" accelerate
import transformers
from transformers.modeling_utils import PreTrainedModel
assert transformers.__version__ == "4.46.3", "pin did not take — restart and re-run"
assert "MODEL_PATH" in globals(), "run §0a first"
print("OK transformers", transformers.__version__)


## 1 · Config

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
import os, glob

DATASET       = "NYC"
DATA_DIR      = "/kaggle/input"
OUT_DIR       = "/kaggle/working"
MODEL_NAME    = "inclusionAI/LLaDA-MoE-7B-A1B-Instruct"

EMB_CONDITION = "hyperbolic"
EMB_FILE      = "/kaggle/input/datasets/yosrkharrat/kushflq/poi_hyperbolic_embs.npy"
CURVATURE_C   = 1.0
PROJ_HIDDEN   = None          # None -> H // 2

TRAIN_FRAC, VAL_FRAC = 0.70, 0.10   # the run-2 re-split (test gets 0.20)

PROFILE_TOP_K, PROFILE_CATS, PROFILE_HOURS = 5, 3, 3
MASK_TOKEN_ID = 156895        # LLaDA-MoE (NOT 126336)
HIST_LEN      = 15
MAX_LEN       = 1024
BATCH_SIZE    = 8             # 16 usually fits on a T4 for eval — halves the wall-clock
SEED          = 42

# ── the quick-comparison knobs ──────────────────────────────────────────────
TEST_MAX      = 2000          # seeded random subsample of the test split (~±2% abs error on Acc@1)

# grid rows: (label, ckpt_key, split, use_profile, scorer)
#   split: "stage0" = original 80/10/10 CSVs | "resplit" = 70/10/20 per-user chronological
#   scorer: "ckpt" = trained output layer from poi_head.pt | "lm_head" = no output layer
GRID = [
    ("80/10/10, no profile, no output layer",  "full", "stage0",  False, "lm_head"),
    ("70/10/20, no profile, no output layer",  "full", "resplit", False, "lm_head"),
    ("70/10/20, + profile, no output layer",   "full", "resplit", True,  "lm_head"),
    ("70/10/20, + profile, + output layer",    "full", "resplit", True,  "ckpt"),
]
# No bare run-1 checkpoint exists on HF (verified 25 Jul) — the rows above are the comparison.
# To also compare the earlier same-config job, uncomment it in CKPTS_0A and add rows with ckpt="run2-jul16".

def find(fname):
    if os.path.isabs(fname) and os.path.exists(fname):
        return fname
    hits = glob.glob(os.path.join(DATA_DIR, "**", fname), recursive=True)
    if not hits:
        stem, ext = os.path.splitext(os.path.basename(fname))
        hits = sorted(glob.glob(os.path.join(DATA_DIR, "**", f"{stem}*{ext}"), recursive=True))
    assert hits, f"File not found under {DATA_DIR}: {fname}"
    return hits[0]


## 2 · Data + BOTH splits' test examples (stage-0 80/10/10 and re-split 70/10/20)

In [ ]:
# ── Load data, build test examples under both split protocols ───────────────
import numpy as np, pandas as pd, pickle, torch, random
from collections import Counter

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

train_df = pd.read_csv(find(f"train_{DATASET}.csv"))
val_df   = pd.read_csv(find(f"val_{DATASET}.csv"))
test_df  = pd.read_csv(find(f"test_{DATASET}.csv"))
meta_df  = pd.read_csv(find(f"poi_metadata_{DATASET}.csv"))
with open(find("vocab.pkl"), "rb") as f:
    vocab = pickle.load(f)
poi_embs = np.load(find(EMB_FILE))
N_POI = len(meta_df)
assert poi_embs.shape[0] == N_POI
print(f"POIs: {N_POI} | emb dim: {poi_embs.shape[1]} | condition: {EMB_CONDITION}")

for df in (train_df, val_df, test_df):
    ts = pd.to_datetime(df["utc_time"], errors="coerce", utc=True)
    df["hour"] = ts.dt.hour.fillna(12).astype(int)
    df["dow"]  = ts.dt.day_name().fillna("Monday")
poi_cat = meta_df.set_index("poi_idx")["category"].fillna("Venue").to_dict()

for _df, _name in ((train_df, "train"), (val_df, "val"), (test_df, "test")):
    _df["split"] = _name
full_stage0 = pd.concat([train_df, val_df, test_df], ignore_index=True)

def resplit_per_user(df, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    df = df.sort_values(["user_id", "utc_time"], kind="mergesort").reset_index(drop=True)
    splits = np.empty(len(df), dtype=object)
    for _, idx in df.groupby("user_id", sort=False).indices.items():
        n = len(idx)
        n_tr  = max(1, int(np.ceil(n * train_frac)))
        n_val = min(max(int(np.ceil(n * (train_frac + val_frac))), n_tr), n)
        splits[idx[:n_tr]] = "train"; splits[idx[n_tr:n_val]] = "val"; splits[idx[n_val:]] = "test"
    out = df.copy(); out["split"] = splits
    return out

full_resplit = resplit_per_user(full_stage0)

def _profile_from_prefix(poi_c, cat_c, hr_c, n_seen):
    return dict(n_seen=n_seen,
                top_pois=poi_c.most_common(PROFILE_TOP_K),
                top_cats=[c for c, _ in cat_c.most_common(PROFILE_CATS)],
                top_hrs=[h for h, _ in hr_c.most_common(PROFILE_HOURS)])

def build_split_examples(full_df, target_split, hist_len=HIST_LEN):
    ex = []
    for uid, g in full_df.groupby("user_id"):
        g = g.sort_values("utc_time")
        seq, hrs = g["poi_idx"].tolist(), g["hour"].tolist()
        dows, splt = g["dow"].tolist(), g["split"].tolist()
        poi_c, cat_c, hr_c = Counter(), Counter(), Counter()
        poi_c[seq[0]] += 1; cat_c[poi_cat.get(seq[0], "Venue")] += 1; hr_c[hrs[0]] += 1
        for i in range(1, len(seq)):
            if splt[i] == target_split:
                ex.append(dict(user=uid, hist=seq[max(0, i-hist_len):i],
                               hist_hours=hrs[max(0, i-hist_len):i],
                               profile=_profile_from_prefix(poi_c, cat_c, hr_c, i),
                               target=seq[i], t_hour=hrs[i], t_dow=dows[i]))
            poi_c[seq[i]] += 1; cat_c[poi_cat.get(seq[i], "Venue")] += 1; hr_c[hrs[i]] += 1
    return ex

def subsample(ex, n, seed=SEED):
    if len(ex) <= n: return ex
    rng = random.Random(seed)
    return rng.sample(ex, n)          # seeded -> identical subset across all grid rows

TEST_EX = {
    "stage0":  subsample(build_split_examples(full_stage0,  "test"), TEST_MAX),
    "resplit": subsample(build_split_examples(full_resplit, "test"), TEST_MAX),
}
for k, v in TEST_EX.items():
    print(f"test[{k}]: {len(v)} examples (subsampled from full split, seed={SEED})")


## 3 · Injection utilities, compat shim, model load, frozen `W_POI` — same as Stage 6b

In [ ]:
# ── logmap0 + injection ─────────────────────────────────────────────────────
import torch

def logmap0(x, c=1.0, eps=1e-9):
    sqrt_c = c ** 0.5
    norm = x.norm(dim=-1, keepdim=True).clamp_min(eps)
    max_norm = (1.0 - 1e-5) / sqrt_c
    x = torch.where(norm > max_norm, x / norm * max_norm, x)
    norm = x.norm(dim=-1, keepdim=True).clamp_min(eps)
    return torch.atanh((sqrt_c * norm).clamp(max=1 - 1e-7)) * x / (sqrt_c * norm)

def build_injection(embs_np, d_model, condition, target_row_norm, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    if condition == "random":
        inj = torch.randn(embs_np.shape[0], d_model, generator=g)
    else:
        e = torch.tensor(embs_np, dtype=torch.float32)
        v = logmap0(e, c=CURVATURE_C) if condition == "hyperbolic" else e
        W = torch.randn(v.shape[1], d_model, generator=g) / (v.shape[1] ** 0.5)
        inj = v @ W
    return inj / inj.norm(dim=-1, keepdim=True).clamp_min(1e-9) * target_row_norm

import transformers.modeling_utils as _mu
if not hasattr(_mu.PreTrainedModel, "all_tied_weights_keys"):
    _mu.PreTrainedModel.all_tied_weights_keys = {}
print("utilities + compat shim ready")


In [ ]:
# ── Model + tokenizer + vocab extension + frozen W_POI ─────────────────────
import gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
import torch, torch.nn as nn

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    assert free > 6e9, f"only {free/1e9:.1f} GiB free — restart the SESSION"

_bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if _bf16_ok else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,
                         bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModel.from_pretrained(MODEL_PATH, trust_remote_code=True,
                                  quantization_config=bnb, device_map={"": 0})

poi_tokens = [f"<poi_{i}>" for i in range(N_POI)]
tokenizer.add_tokens(poi_tokens, special_tokens=True)
model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
POI_TOKEN_IDS = torch.tensor(tokenizer.convert_tokens_to_ids(poi_tokens))
POI_ID_START, POI_ID_END = int(POI_TOKEN_IDS.min()), int(POI_TOKEN_IDS.max())
assert POI_ID_END - POI_ID_START + 1 == N_POI
print(f"POI id range: [{POI_ID_START}, {POI_ID_END}]")

base_emb  = model.get_input_embeddings()
d_model   = base_emb.weight.shape[1]
emb_device = base_emb.weight.device
with torch.no_grad():
    tgt_norm = base_emb.weight[:POI_ID_START].float().norm(dim=-1).mean().item()
    inj = build_injection(poi_embs, d_model, EMB_CONDITION, tgt_norm)

W_POI = nn.Embedding(N_POI, d_model, dtype=torch.float16, device=emb_device)
with torch.no_grad():
    W_POI.weight.copy_(inj.to(torch.float16))
W_POI.weight.requires_grad_(False)

_orig_emb_forward = base_emb.forward
def _mixed_embedding_forward(input_ids):
    is_poi = (input_ids >= POI_ID_START) & (input_ids <= POI_ID_END)
    out = _orig_emb_forward(torch.where(is_poi, torch.zeros_like(input_ids), input_ids))
    if is_poi.any():
        out = out.clone()
        out[is_poi] = W_POI((input_ids[is_poi] - POI_ID_START).clamp_(0, N_POI - 1)).to(out.dtype)
    return out
base_emb.forward = _mixed_embedding_forward
print(f"W_POI {tuple(W_POI.weight.shape)} frozen; mixed-embedding wrapper installed")


## 4 · Load ALL adapters (multi-adapter PEFT) + per-checkpoint scorers

Each checkpoint's LoRA is loaded under its key; `set_active(key)` switches adapter **and** scorer.
A checkpoint without `poi_head.pt` (the bare experiment) gets `scorer=None` — only `lm_head`
rows are valid for it.

In [ ]:
# ── adapters + scorers ──────────────────────────────────────────────────────
from peft import PeftModel
import torch.nn as nn

keys = list(CKPT_DIRS.keys())
model = PeftModel.from_pretrained(model, CKPT_DIRS[keys[0]], adapter_name=keys[0], is_trainable=False)
for k in keys[1:]:
    model.load_adapter(CKPT_DIRS[k], adapter_name=k, is_trainable=False)
model.eval()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

H = model.config.hidden_size
PROJ_H = PROJ_HIDDEN or (H // 2)

def build_scorer(ckpt_dir):
    p = os.path.join(ckpt_dir, "poi_head.pt")
    if not os.path.isfile(p):
        return None
    ck = torch.load(p, map_location="cpu")
    mode = ck.get("scoring_mode", "both")
    norm = nn.LayerNorm(H);  norm.load_state_dict(ck["poi_norm"])
    head = nn.Linear(H, N_POI, bias=True); head.load_state_dict(ck["poi_head"])
    proj = nn.Sequential(nn.Linear(H, PROJ_H), nn.GELU(), nn.LayerNorm(PROJ_H),
                         nn.Linear(PROJ_H, d_model, bias=False))
    proj.load_state_dict(ck["poi_proj"])
    for m in (norm, head, proj):
        m.to(emb_device, dtype=torch.float32).eval()
        for q in m.parameters(): q.requires_grad_(False)
    assert head.weight.abs().sum().item() > 0
    return dict(mode=mode, norm=norm, head=head, proj=proj)

SCORERS = {k: build_scorer(d) for k, d in CKPT_DIRS.items()}
for k, s in SCORERS.items():
    print(f"[{k}] scorer: {'poi_head.pt mode=' + s['mode'] if s else 'NONE (bare — lm_head only)'}")

_W_POI_f = W_POI.weight.detach().float()
ACTIVE = {"key": None}

def set_active(key):
    model.set_adapter(key)
    ACTIVE["key"] = key

def ckpt_scores(h):
    s = SCORERS[ACTIVE["key"]]
    assert s is not None, f"checkpoint '{ACTIVE['key']}' has no output layer — use scorer='lm_head'"
    h = s["norm"](h)
    if s["mode"] == "head":  return s["head"](h)
    tied = s["proj"](h) @ _W_POI_f.t()
    return tied if s["mode"] == "tied" else s["head"](h) + tied


## 5 · Prompt (profile ON/OFF) + collator + the two scoring paths

In [ ]:
# ── prompts + collator + eval ───────────────────────────────────────────────
from torch.utils.data import DataLoader
import time

EOS_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else (tokenizer.pad_token_id or 0)

def profile_text(pr):
    if not pr["top_pois"]:
        return "[user profile]\nnew user, no prior check-ins\n"
    pois = ", ".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, x{c})" for p, c in pr["top_pois"])
    lines = ["[user profile]", f"check-ins so far: {pr['n_seen']}", f"most visited: {pois}"]
    if pr["top_cats"]: lines.append("favourite categories: " + ", ".join(map(str, pr["top_cats"])))
    if pr["top_hrs"]:  lines.append("usual hours: " + ", ".join(f"{h}:00" for h in sorted(pr["top_hrs"])))
    return "\n".join(lines) + "\n"

def prompt_text(ex, use_profile):
    hist = "\n".join(f"<poi_{p}> ({poi_cat.get(p,'Venue')}, {h}:00)"
                     for p, h in zip(ex["hist"], ex["hist_hours"]))
    head = ("You are a POI recommendation expert. Using the user's long-term profile and their "
            "recent check-ins, predict the next POI token.\n") if use_profile else \
           ("You are a POI recommendation expert. Using the user's recent check-ins, "
            "predict the next POI token.\n")
    prof = profile_text(ex["profile"]) if use_profile else ""
    return (head + prof + "[recent check-ins]\n" + hist +
            f"\n[current time] {ex['t_dow']} {ex['t_hour']}:00\n[next POI] ")

def collate_fn(use_profile):
    def collate(batch):
        enc = []
        for ex in batch:
            p_ids = tokenizer(prompt_text(ex, use_profile), add_special_tokens=False,
                              truncation=True, max_length=MAX_LEN - 2)["input_ids"]
            enc.append((p_ids, POI_ID_START + ex["target"]))
        L = max(len(p) + 1 for p, _ in enc)
        input_ids = torch.full((len(batch), L), EOS_ID, dtype=torch.long)
        labels    = torch.full((len(batch), L), -100,   dtype=torch.long)
        for i, (p_ids, tgt) in enumerate(enc):
            start = L - (len(p_ids) + 1)
            input_ids[i, start:start+len(p_ids)] = torch.tensor(p_ids)
            input_ids[i, start+len(p_ids)] = MASK_TOKEN_ID
            labels[i, start+len(p_ids)]    = tgt
        return dict(input_ids=input_ids, labels=labels)
    return collate

def _backbone():
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    return base.model

def _lm_head():
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    out_emb = base.get_output_embeddings()
    assert out_emb is not None, "model has no output embedding / lm_head"
    return out_emb

@torch.no_grad()
def evaluate(examples, use_profile, scorer, name):
    dev = next(iter(model.parameters())).device
    loader = DataLoader(examples, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn(use_profile))
    ranks, t0 = [], time.time()
    for bi, batch in enumerate(loader):
        ids, labels = batch["input_ids"].to(dev), batch["labels"].to(dev)
        out = _backbone()(input_ids=ids)
        h = out.last_hidden_state if hasattr(out, "last_hidden_state") else out[0]
        pos = (labels != -100)
        hh = h[pos].float()
        if scorer == "ckpt":
            logits = ckpt_scores(hh)                                   # trained output layer
        else:                                                          # "lm_head": no output layer
            logits = _lm_head()(hh.to(h.dtype)).float()[:, POI_ID_START:POI_ID_END + 1]
        tgt = labels[pos] - POI_ID_START
        r = (logits > logits.gather(1, tgt[:, None])).sum(1) + 1
        ranks.extend(r.tolist())
        if bi % 50 == 0:
            print(f"    [{name}] batch {bi}/{len(loader)} ({time.time()-t0:.0f}s)")
    ranks = torch.tensor(ranks, dtype=torch.float)
    return dict(n=len(ranks),
                acc1=(ranks <= 1).float().mean().item(),
                acc5=(ranks <= 5).float().mean().item(),
                acc10=(ranks <= 10).float().mean().item(),
                mrr=(1.0 / ranks).mean().item())


## 6 · Run the grid → comparison table

In [ ]:
# ── run the comparison grid ─────────────────────────────────────────────────
import json, pandas as pd

rows = []
for label, ckpt_key, split, use_profile, scorer in GRID:
    if SCORERS.get(ckpt_key) is None and scorer == "ckpt":
        print(f"SKIP {label!r}: checkpoint '{ckpt_key}' has no output layer"); continue
    print(f"\n=== {label}  [ckpt={ckpt_key} split={split} profile={use_profile} scorer={scorer}] ===")
    set_active(ckpt_key)
    res = evaluate(TEST_EX[split], use_profile, scorer, label)
    rows.append(dict(config=label, ckpt=ckpt_key, split=split,
                     profile=use_profile, scorer=scorer, **res))
    print(f"  -> Acc@1={res['acc1']:.4f}  Acc@5={res['acc5']:.4f}  "
          f"Acc@10={res['acc10']:.4f}  MRR={res['mrr']:.4f}")
    with open(f"{OUT_DIR}/quick_comparison_{DATASET}.json", "w") as f:
        json.dump(rows, f, indent=2)          # saved after EVERY row — interruptions keep partials

tab = pd.DataFrame(rows)[["config", "n", "acc1", "acc5", "acc10", "mrr"]]
tab.to_csv(f"{OUT_DIR}/quick_comparison_{DATASET}.csv", index=False)
print("\n" + "="*100)
print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nMarkdown (paste into the docs):\n")
print("| Configuration | Acc@1 | Acc@5 | Acc@10 | MRR |")
print("|---|---|---|---|---|")
for r in rows:
    print(f"| {r['config']} | {r['acc1']:.4f} | {r['acc5']:.4f} | {r['acc10']:.4f} | {r['mrr']:.4f} |")


## Notes

- All rows use the **same seeded subsample** per split (`TEST_MAX`, seed 42), so rows are directly
  comparable to each other; expect ±0.02 absolute wobble vs the full test set at n=2000.
- The stage-0 (80/10/10) and re-split (70/10/20) rows use *different* target sets by construction —
  that comparison measures the protocol change itself, target-set shift included.
- On the full checkpoint the `lm_head` rows are a *lower bound* stand-in for "no output layer"
  (those rows were never trained). For the true bare number, run the bare run-1 checkpoint
  (add it in §0a + GRID) — its lm_head POI rows *were* its training target.
